In [42]:
import pandas as pd
import numpy as np

In [43]:
df_diabetes = pd.read_csv("../datasets/raw/diabetes.csv")
df_heart = pd.read_csv("../datasets/raw/heart.csv")
df_liver = pd.read_csv("../datasets/raw/liver.csv", encoding='latin1')
df_kidney = pd.read_csv("../datasets/raw/kidney.csv",encoding='latin1',engine='python',on_bad_lines='skip')

In [44]:
df_diabetes.rename(columns={"Diabetes_binary": "target"}, inplace=True)
# Heart target is created later from two columns
df_liver.rename(columns={"Result": "target"}, inplace=True)
df_kidney.columns.values[25] = "target"

In [45]:
df_liver["target"] = df_liver["target"].replace(2, 0)

In [46]:
cols = ["BMI"]
df_diabetes[cols] = df_diabetes[cols].replace(0, np.nan)
df_diabetes.fillna(df_diabetes.median(), inplace=True)
print("Diabetes dataset cleaned")

Diabetes dataset cleaned


In [47]:
print("Processing Heart Disease Dataset...")

# ── Create binary target ──────────────────────
df_heart['target'] = (
    (df_heart['HadHeartAttack'] == 'Yes') |
    (df_heart['HadAngina'] == 'Yes')
).astype(int)

print(f"Target distribution:\n{df_heart['target'].value_counts()}")

# ── Select relevant features ──────────────────
selected_features = [
    'Sex', 'GeneralHealth', 'PhysicalHealthDays',
    'MentalHealthDays', 'PhysicalActivities', 'SleepHours',
    'HadStroke', 'HadAsthma', 'HadSkinCancer', 'HadCOPD',
    'HadDepressiveDisorder', 'HadKidneyDisease', 'HadArthritis',
    'HadDiabetes', 'DeafOrHardOfHearing', 'BlindOrVisionDifficulty',
    'DifficultyConcentrating', 'DifficultyWalking',
    'DifficultyDressingBathing', 'DifficultyErrands',
    'SmokerStatus', 'ChestScan', 'AgeCategory',
    'BMI', 'AlcoholDrinkers', 'target'
]
df_heart = df_heart[selected_features].copy()

# ── Encode AgeCategory → numeric midpoints ────
age_map = {
    'Age 18 to 24': 21, 'Age 25 to 29': 27,
    'Age 30 to 34': 32, 'Age 35 to 39': 37,
    'Age 40 to 44': 42, 'Age 45 to 49': 47,
    'Age 50 to 54': 52, 'Age 55 to 59': 57,
    'Age 60 to 64': 62, 'Age 65 to 69': 67,
    'Age 70 to 74': 72, 'Age 75 to 79': 77,
    'Age 80 or older': 82
}
df_heart['AgeCategory'] = df_heart['AgeCategory'].map(age_map)

# ── Encode Sex ────────────────────────────────
df_heart['Sex'] = df_heart['Sex'].map({'Male': 1, 'Female': 0})

# ── Encode Yes/No binary columns ──────────────
yes_no_cols = [
    'PhysicalActivities', 'HadStroke', 'HadAsthma',
    'HadSkinCancer', 'HadCOPD', 'HadDepressiveDisorder',
    'HadKidneyDisease', 'HadArthritis', 'DeafOrHardOfHearing',
    'BlindOrVisionDifficulty', 'DifficultyConcentrating',
    'DifficultyWalking', 'DifficultyDressingBathing',
    'DifficultyErrands', 'ChestScan', 'AlcoholDrinkers'
]
for col in yes_no_cols:
    df_heart[col] = df_heart[col].map({'Yes': 1, 'No': 0})

# ── Encode GeneralHealth ──────────────────────
health_map = {
    'Excellent': 5, 'Very good': 4,
    'Good': 3, 'Fair': 2, 'Poor': 1
}
df_heart['GeneralHealth'] = df_heart['GeneralHealth'].map(health_map)

# ── Encode HadDiabetes ────────────────────────
diabetes_map = {
    'No': 0,
    'No, pre-diabetes or borderline diabetes': 0,
    'Yes': 1,
    'Yes, but only during pregnancy (female)': 0
}
df_heart['HadDiabetes'] = df_heart['HadDiabetes'].map(diabetes_map)

# ── Encode SmokerStatus ───────────────────────
smoker_map = {
    'Never smoked': 0,
    'Former smoker': 1,
    'Current smoker - now smokes some days': 2,
    'Current smoker - now smokes every day': 3
}
df_heart['SmokerStatus'] = df_heart['SmokerStatus'].map(smoker_map)

# ── Drop any nulls from mapping mismatches ────
df_heart = df_heart.dropna()
df_heart = df_heart.drop_duplicates()

print(f"Final shape: {df_heart.shape}")
print(f"Null check:\n{df_heart.isnull().sum().sum()} nulls remaining")
print(f"Dtypes:\n{df_heart.dtypes}")
print("Heart dataset cleaned")

Processing Heart Disease Dataset...
Target distribution:
target
0    224406
1     21616
Name: count, dtype: int64


Final shape: (242466, 26)
Null check:
0 nulls remaining
Dtypes:
Sex                            int64
GeneralHealth                  int64
PhysicalHealthDays           float64
MentalHealthDays             float64
PhysicalActivities             int64
SleepHours                   float64
HadStroke                      int64
HadAsthma                      int64
HadSkinCancer                  int64
HadCOPD                        int64
HadDepressiveDisorder          int64
HadKidneyDisease               int64
HadArthritis                   int64
HadDiabetes                    int64
DeafOrHardOfHearing            int64
BlindOrVisionDifficulty        int64
DifficultyConcentrating        int64
DifficultyWalking              int64
DifficultyDressingBathing      int64
DifficultyErrands              int64
SmokerStatus                   int64
ChestScan                      int64
AgeCategory                    int64
BMI                          float64
AlcoholDrinkers                int64
target     

In [48]:
df_liver.columns = df_liver.columns.str.strip()
df_liver.rename(columns={"Gender of the patient": "Gender"}, inplace=True)
df_liver.columns = df_liver.columns.str.strip()
df_liver["Gender"] = df_liver["Gender"].map({"Male": 1, "Female": 0})
df_liver.fillna(df_liver.median(), inplace=True)
print("Liver dataset cleaned")

Liver dataset cleaned


In [49]:
df_kidney.columns = [col.strip("'").strip() for col in df_kidney.columns]

df_kidney = df_kidney.drop(columns=['id'], errors='ignore')

In [50]:
df_kidney.replace("?", np.nan, inplace=True)
df_kidney = df_kidney.apply(lambda x: x.str.strip() if x.dtype == "object" else x)
df_kidney["target"] = df_kidney["target"].replace({"ckd": 1,"notckd": 0,"ckd\t": 1})

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
for col in df_kidney.select_dtypes(include="object").columns:
    df_kidney[col] = df_kidney[col].fillna(df_kidney[col].mode()[0])
    df_kidney[col] = le.fit_transform(df_kidney[col].astype(str))
df_kidney.fillna(df_kidney.median(), inplace=True)
print("Kidney dataset cleaned")

C:\Users\prajw\AppData\Local\Temp\ipykernel_3012\3799502873.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df_kidney.select_dtypes(include="object").columns:


Kidney dataset cleaned


In [51]:
print("Diabetes:\n", df_diabetes.isnull().sum())
print("Heart:\n", df_heart.isnull().sum())
print("Liver:\n", df_liver.isnull().sum())
print("Kidney:\n", df_kidney.isnull().sum())

Diabetes:
 target                  0
HighBP                  0
HighChol                0
CholCheck               0
BMI                     0
Smoker                  0
Stroke                  0
HeartDiseaseorAttack    0
PhysActivity            0
Fruits                  0
Veggies                 0
HvyAlcoholConsump       0
AnyHealthcare           0
NoDocbcCost             0
GenHlth                 0
MentHlth                0
PhysHlth                0
DiffWalk                0
Sex                     0
Age                     0
Education               0
Income                  0
dtype: int64
Heart:
 Sex                          0
GeneralHealth                0
PhysicalHealthDays           0
MentalHealthDays             0
PhysicalActivities           0
SleepHours                   0
HadStroke                    0
HadAsthma                    0
HadSkinCancer                0
HadCOPD                      0
HadDepressiveDisorder        0
HadKidneyDisease             0
HadArthritis            

In [52]:
print(df_diabetes.dtypes)
print(df_heart.dtypes)
print(df_liver.dtypes)
print(df_kidney.dtypes)

target                  float64
HighBP                  float64
HighChol                float64
CholCheck               float64
BMI                     float64
Smoker                  float64
Stroke                  float64
HeartDiseaseorAttack    float64
PhysActivity            float64
Fruits                  float64
Veggies                 float64
HvyAlcoholConsump       float64
AnyHealthcare           float64
NoDocbcCost             float64
GenHlth                 float64
MentHlth                float64
PhysHlth                float64
DiffWalk                float64
Sex                     float64
Age                     float64
Education               float64
Income                  float64
dtype: object
Sex                            int64
GeneralHealth                  int64
PhysicalHealthDays           float64
MentalHealthDays             float64
PhysicalActivities             int64
SleepHours                   float64
HadStroke                      int64
HadAsthma              

In [53]:
df_diabetes.to_csv("../datasets/processed/diabetes_clean.csv", index=False)
df_heart.to_csv("../datasets/processed/heart_clean.csv", index=False)
df_liver.to_csv("../datasets/processed/liver_clean.csv", index=False)
df_kidney.to_csv("../datasets/processed/kidney_clean.csv", index=False)
print("All cleaned datasets saved")

All cleaned datasets saved
